In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os

from ADFWI.utils.assessment_metric import MAPE, MSE, SSIM

base_path = "../Inductive_bias/regularization/Marmousi2-nowater/GC-GaussianNoise-for-all-Smooth=6"

init_model = np.load(os.path.join(base_path,"data-mean=1-std=1/model/init_model.npz"))
true_model = np.load(os.path.join(base_path,"data-mean=1-std=1/model/true_model.npz"))
init_v   = init_model["vp"]
init_rho = init_model["rho"]
true_v   = true_model["vp"]
true_rho = true_model["rho"]

ox, oz  = 0, 0        
nz, nx  = 76, 200      
dx, dz  = 40, 40         
nt, dt  = 2500, 0.003     
nabc    = 30                 
x       = np.arange(nx)*dx/1000
z       = np.arange(nz)*dz/1000
x_mesh,z_mesh = np.meshgrid(x,z)
src_z = np.array([1  for i in range(2,nx-1,5)])*dz/1000
src_x = np.array([i  for i in range(2,nx-1,5)])*dx/1000
rcv_z = np.array([1  for i in range(0,nx,1)])*dz/1000
rcv_x = np.array([j  for j in range(0,nx,1)])*dz/1000
vmin = true_v.min();vmax = true_v.max()  

MAX_ITER = 500

In [ ]:
import matplotlib.transforms as mtransforms
from scipy.interpolate import griddata

def plot_vel_single_for_all(fig,ax,v,title="",MSE="",vmin=None,vmax=None,cmap = "rainbow"):
    plt.rc('font',family='Times New Roman')
    # plm = ax.pcolormesh(x_mesh, z_mesh, v,cmap=cmap,vmin=vmin,vmax=vmax)
    x = np.arange(nx*3)*dx/3/1000
    z = np.arange(nz*3)*dz/3/1000
    x_mesh_new,z_mesh_new = np.meshgrid(x,z)

    v_new = griddata((x_mesh.flatten(), z_mesh.flatten()), v.flatten(), (x_mesh_new, z_mesh_new), method='cubic')
    
    plm = ax.pcolormesh(x_mesh_new, z_mesh_new, v_new,cmap=cmap,vmin=vmin,vmax=vmax,shading="nearest")
    ax.invert_yaxis()
    ax.tick_params(labelsize = 14)
    ax.set_title(title,fontsize=14)
    ax.text(0.2,0.4,MSE,fontsize=14,c="w")
    return plm

def add_right_cax(ax, pad, width):
    axpos = ax.get_position()
    caxpos = mtransforms.Bbox.from_extents(
        axpos.x1 + pad,
        axpos.y0,
        axpos.x1 + pad + width,
        axpos.y1
    )
    cax = ax.figure.add_axes(caxpos)

    return cax

def add_bottom_cax(ax, pad, height):
    axpos = ax.get_position()
    caxpos = mtransforms.Bbox.from_extents(
        axpos.x0,
        axpos.y0 - pad - height,
        axpos.x1,
        axpos.y0 - pad
    )
    cax = ax.figure.add_axes(caxpos)
    
    return cax

def plot_vel_singleline_for_all(ax,v_true,v_init,v_inv,x_distance,title,show_xlabel=True,show_ylabel=False,show_legend=False):
    ax.plot(v_true[:,int(x_distance//dx)]/1000,  z, c='k',   linewidth=2, linestyle="-" ,label="True")
    ax.plot(v_init[:,int(x_distance//dx)]/1000,  z, c='gray',linewidth=2, linestyle="-" ,label="Init")
    ax.plot(v_inv [:,int(x_distance//dx)]/1000,  z, c='r',   linewidth=2, linestyle="--",label="Inverted")
    ax.tick_params(labelsize = 12)
    if not show_xlabel:
        ax.set_xticks([])

    if not show_ylabel:
        ax.set_yticks([])
    else:
        ax.tick_params(labelsize = 10)
    ax.invert_yaxis()
    
    if show_legend:
        ax.legend(fontsize = 12)
    ax.set_title(title,fontsize=12)

In [ ]:
itervp_baseline0    = np.load(os.path.join(base_path,"data-mean=0-std=0/inversion-vp-baseline/iter_vp.npz"))["data"][:MAX_ITER]
itervp_baseline1    = np.load(os.path.join(base_path,"data-mean=1-std=1/inversion-vp-baseline/iter_vp.npz"))["data"][:MAX_ITER]
itervp_baseline2    = np.load(os.path.join(base_path,"data-mean=1-std=2/inversion-vp-baseline/iter_vp.npz"))["data"][:MAX_ITER]
itervp_baseline3    = np.load(os.path.join(base_path,"data-mean=1-std=3/inversion-vp-baseline/iter_vp.npz"))["data"][:MAX_ITER]
itervp_baseline4    = np.load(os.path.join(base_path,"data-mean=1-std=4/inversion-vp-baseline/iter_vp.npz"))["data"][:MAX_ITER]
itervp_baseline5    = np.load(os.path.join(base_path,"data-mean=1-std=5/inversion-vp-baseline/iter_vp.npz"))["data"][:MAX_ITER]
itervp_baseline6    = np.load(os.path.join(base_path,"data-mean=1-std=6/inversion-vp-baseline/iter_vp.npz"))["data"][:MAX_ITER]

itervp_CNN_1x64_0   = np.load(os.path.join(base_path,"data-mean=0-std=0/inversion-vp-CNN-1x64-v/iter_vp.npz"))["data"][:MAX_ITER]
itervp_CNN_1x64_1   = np.load(os.path.join(base_path,"data-mean=1-std=1/inversion-vp-CNN-1x64-v/iter_vp.npz"))["data"][:MAX_ITER]
itervp_CNN_1x64_2   = np.load(os.path.join(base_path,"data-mean=1-std=2/inversion-vp-CNN-1x64-v/iter_vp.npz"))["data"][:MAX_ITER]
itervp_CNN_1x64_3   = np.load(os.path.join(base_path,"data-mean=1-std=3/inversion-vp-CNN-1x64-v/iter_vp.npz"))["data"][:MAX_ITER]
itervp_CNN_1x64_4   = np.load(os.path.join(base_path,"data-mean=1-std=4/inversion-vp-CNN-1x64-v/iter_vp.npz"))["data"][:MAX_ITER]
itervp_CNN_1x64_5   = np.load(os.path.join(base_path,"data-mean=1-std=5/inversion-vp-CNN-1x64-v/iter_vp.npz"))["data"][:MAX_ITER]
itervp_CNN_1x64_6   = np.load(os.path.join(base_path,"data-mean=1-std=6/inversion-vp-CNN-1x64-v/iter_vp.npz"))["data"][:MAX_ITER]

itervp_CNN_1x128_0   = np.load(os.path.join(base_path,"data-mean=0-std=0/inversion-vp-CNN-1x128-v/iter_vp.npz"))["data"][:MAX_ITER]
itervp_CNN_1x128_1   = np.load(os.path.join(base_path,"data-mean=1-std=1/inversion-vp-CNN-1x128-v/iter_vp.npz"))["data"][:MAX_ITER]
itervp_CNN_1x128_2   = np.load(os.path.join(base_path,"data-mean=1-std=2/inversion-vp-CNN-1x128-v/iter_vp.npz"))["data"][:MAX_ITER]
itervp_CNN_1x128_3   = np.load(os.path.join(base_path,"data-mean=1-std=3/inversion-vp-CNN-1x128-v/iter_vp.npz"))["data"][:MAX_ITER]
itervp_CNN_1x128_4   = np.load(os.path.join(base_path,"data-mean=1-std=4/inversion-vp-CNN-1x128-v/iter_vp.npz"))["data"][:MAX_ITER]
itervp_CNN_1x128_5   = np.load(os.path.join(base_path,"data-mean=1-std=5/inversion-vp-CNN-1x128-v/iter_vp.npz"))["data"][:MAX_ITER]
itervp_CNN_1x128_6   = np.load(os.path.join(base_path,"data-mean=1-std=6/inversion-vp-CNN-1x128-v/iter_vp.npz"))["data"][:MAX_ITER]

itervp_CNN_1x256_0   = np.load(os.path.join(base_path,"data-mean=0-std=0/inversion-vp-CNN-1x256-v/iter_vp.npz"))["data"][:MAX_ITER]
itervp_CNN_1x256_1   = np.load(os.path.join(base_path,"data-mean=1-std=1/inversion-vp-CNN-1x256-v/iter_vp.npz"))["data"][:MAX_ITER]
itervp_CNN_1x256_2   = np.load(os.path.join(base_path,"data-mean=1-std=2/inversion-vp-CNN-1x256-v/iter_vp.npz"))["data"][:MAX_ITER]
itervp_CNN_1x256_3   = np.load(os.path.join(base_path,"data-mean=1-std=3/inversion-vp-CNN-1x256-v/iter_vp.npz"))["data"][:MAX_ITER]
itervp_CNN_1x256_4   = np.load(os.path.join(base_path,"data-mean=1-std=4/inversion-vp-CNN-1x256-v/iter_vp.npz"))["data"][:MAX_ITER]
itervp_CNN_1x256_5   = np.load(os.path.join(base_path,"data-mean=1-std=5/inversion-vp-CNN-1x256-v/iter_vp.npz"))["data"][:MAX_ITER]
itervp_CNN_1x256_6   = np.load(os.path.join(base_path,"data-mean=1-std=6/inversion-vp-CNN-1x256-v/iter_vp.npz"))["data"][:MAX_ITER]

itervp_CNN_2x64_0  = np.load(os.path.join(base_path,"data-mean=0-std=0/inversion-vp-CNN-2x64-v/iter_vp.npz"))["data"][:MAX_ITER]
itervp_CNN_2x64_1  = np.load(os.path.join(base_path,"data-mean=1-std=1/inversion-vp-CNN-2x64-v/iter_vp.npz"))["data"][:MAX_ITER]
itervp_CNN_2x64_2  = np.load(os.path.join(base_path,"data-mean=1-std=2/inversion-vp-CNN-2x64-v/iter_vp.npz"))["data"][:MAX_ITER]
itervp_CNN_2x64_3  = np.load(os.path.join(base_path,"data-mean=1-std=3/inversion-vp-CNN-2x64-v/iter_vp.npz"))["data"][:MAX_ITER]
itervp_CNN_2x64_4  = np.load(os.path.join(base_path,"data-mean=1-std=4/inversion-vp-CNN-2x64-v/iter_vp.npz"))["data"][:MAX_ITER]
itervp_CNN_2x64_5  = np.load(os.path.join(base_path,"data-mean=1-std=5/inversion-vp-CNN-2x64-v/iter_vp.npz"))["data"][:MAX_ITER]
itervp_CNN_2x64_6  = np.load(os.path.join(base_path,"data-mean=1-std=6/inversion-vp-CNN-2x64-v/iter_vp.npz"))["data"][:MAX_ITER]

itervp_CNN_2x128_0  = np.load(os.path.join(base_path,"data-mean=0-std=0/inversion-vp-CNN-2x128-v/iter_vp.npz"))["data"][:MAX_ITER]
itervp_CNN_2x128_1  = np.load(os.path.join(base_path,"data-mean=1-std=1/inversion-vp-CNN-2x128-v/iter_vp.npz"))["data"][:MAX_ITER]
itervp_CNN_2x128_2  = np.load(os.path.join(base_path,"data-mean=1-std=2/inversion-vp-CNN-2x128-v/iter_vp.npz"))["data"][:MAX_ITER]
itervp_CNN_2x128_3  = np.load(os.path.join(base_path,"data-mean=1-std=3/inversion-vp-CNN-2x128-v/iter_vp.npz"))["data"][:MAX_ITER]
itervp_CNN_2x128_4  = np.load(os.path.join(base_path,"data-mean=1-std=4/inversion-vp-CNN-2x128-v/iter_vp.npz"))["data"][:MAX_ITER]
itervp_CNN_2x128_5  = np.load(os.path.join(base_path,"data-mean=1-std=5/inversion-vp-CNN-2x128-v/iter_vp.npz"))["data"][:MAX_ITER]
itervp_CNN_2x128_6  = np.load(os.path.join(base_path,"data-mean=1-std=6/inversion-vp-CNN-2x128-v/iter_vp.npz"))["data"][:MAX_ITER]

itervp_CNN_2x256_0  = np.load(os.path.join(base_path,"data-mean=0-std=0/inversion-vp-CNN-2x256-v/iter_vp.npz"))["data"][:MAX_ITER]
itervp_CNN_2x256_1  = np.load(os.path.join(base_path,"data-mean=1-std=1/inversion-vp-CNN-2x256-v/iter_vp.npz"))["data"][:MAX_ITER]
itervp_CNN_2x256_2  = np.load(os.path.join(base_path,"data-mean=1-std=2/inversion-vp-CNN-2x256-v/iter_vp.npz"))["data"][:MAX_ITER]
itervp_CNN_2x256_3  = np.load(os.path.join(base_path,"data-mean=1-std=3/inversion-vp-CNN-2x256-v/iter_vp.npz"))["data"][:MAX_ITER]
itervp_CNN_2x256_4  = np.load(os.path.join(base_path,"data-mean=1-std=4/inversion-vp-CNN-2x256-v/iter_vp.npz"))["data"][:MAX_ITER]
itervp_CNN_2x256_5  = np.load(os.path.join(base_path,"data-mean=1-std=5/inversion-vp-CNN-2x256-v/iter_vp.npz"))["data"][:MAX_ITER]
itervp_CNN_2x256_6  = np.load(os.path.join(base_path,"data-mean=1-std=6/inversion-vp-CNN-2x256-v/iter_vp.npz"))["data"][:MAX_ITER]

itervp_CNN_3x64_0  = np.load(os.path.join(base_path,"data-mean=0-std=0/inversion-vp-CNN-3x64-v/iter_vp.npz"))["data"][:MAX_ITER]
itervp_CNN_3x64_1  = np.load(os.path.join(base_path,"data-mean=1-std=1/inversion-vp-CNN-3x64-v/iter_vp.npz"))["data"][:MAX_ITER]
itervp_CNN_3x64_2  = np.load(os.path.join(base_path,"data-mean=1-std=2/inversion-vp-CNN-3x64-v/iter_vp.npz"))["data"][:MAX_ITER]
itervp_CNN_3x64_3  = np.load(os.path.join(base_path,"data-mean=1-std=3/inversion-vp-CNN-3x64-v/iter_vp.npz"))["data"][:MAX_ITER]
itervp_CNN_3x64_4  = np.load(os.path.join(base_path,"data-mean=1-std=4/inversion-vp-CNN-3x64-v/iter_vp.npz"))["data"][:MAX_ITER]
itervp_CNN_3x64_5  = np.load(os.path.join(base_path,"data-mean=1-std=5/inversion-vp-CNN-3x64-v/iter_vp.npz"))["data"][:MAX_ITER]
itervp_CNN_3x64_6  = np.load(os.path.join(base_path,"data-mean=1-std=6/inversion-vp-CNN-3x64-v/iter_vp.npz"))["data"][:MAX_ITER]

itervp_CNN_3x128_0  = np.load(os.path.join(base_path,"data-mean=0-std=0/inversion-vp-CNN-3x128-v/iter_vp.npz"))["data"][:MAX_ITER]
itervp_CNN_3x128_1  = np.load(os.path.join(base_path,"data-mean=1-std=1/inversion-vp-CNN-3x128-v/iter_vp.npz"))["data"][:MAX_ITER]
itervp_CNN_3x128_2  = np.load(os.path.join(base_path,"data-mean=1-std=2/inversion-vp-CNN-3x128-v/iter_vp.npz"))["data"][:MAX_ITER]
itervp_CNN_3x128_3  = np.load(os.path.join(base_path,"data-mean=1-std=3/inversion-vp-CNN-3x128-v/iter_vp.npz"))["data"][:MAX_ITER]
itervp_CNN_3x128_4  = np.load(os.path.join(base_path,"data-mean=1-std=4/inversion-vp-CNN-3x128-v/iter_vp.npz"))["data"][:MAX_ITER]
itervp_CNN_3x128_5  = np.load(os.path.join(base_path,"data-mean=1-std=5/inversion-vp-CNN-3x128-v/iter_vp.npz"))["data"][:MAX_ITER]
itervp_CNN_3x128_6  = np.load(os.path.join(base_path,"data-mean=1-std=6/inversion-vp-CNN-3x128-v/iter_vp.npz"))["data"][:MAX_ITER]

itervp_CNN_3x256_0  = np.load(os.path.join(base_path,"data-mean=0-std=0/inversion-vp-CNN-3x256-v/iter_vp.npz"))["data"][:MAX_ITER]
itervp_CNN_3x256_1  = np.load(os.path.join(base_path,"data-mean=1-std=1/inversion-vp-CNN-3x256-v/iter_vp.npz"))["data"][:MAX_ITER]
itervp_CNN_3x256_2  = np.load(os.path.join(base_path,"data-mean=1-std=2/inversion-vp-CNN-3x256-v/iter_vp.npz"))["data"][:MAX_ITER]
itervp_CNN_3x256_3  = np.load(os.path.join(base_path,"data-mean=1-std=3/inversion-vp-CNN-3x256-v/iter_vp.npz"))["data"][:MAX_ITER]
itervp_CNN_3x256_4  = np.load(os.path.join(base_path,"data-mean=1-std=4/inversion-vp-CNN-3x256-v/iter_vp.npz"))["data"][:MAX_ITER]
itervp_CNN_3x256_5  = np.load(os.path.join(base_path,"data-mean=1-std=5/inversion-vp-CNN-3x256-v/iter_vp.npz"))["data"][:MAX_ITER]
itervp_CNN_3x256_6  = np.load(os.path.join(base_path,"data-mean=1-std=6/inversion-vp-CNN-3x256-v/iter_vp.npz"))["data"][:MAX_ITER]

In [ ]:
iterloss_baseline0  = np.load(os.path.join(base_path,"data-mean=0-std=0/inversion-vp-baseline/iter_loss.npz"))["data"][:MAX_ITER]
iterloss_baseline1  = np.load(os.path.join(base_path,"data-mean=1-std=1/inversion-vp-baseline/iter_loss.npz"))["data"][:MAX_ITER]
iterloss_baseline2  = np.load(os.path.join(base_path,"data-mean=1-std=2/inversion-vp-baseline/iter_loss.npz"))["data"][:MAX_ITER]
iterloss_baseline3  = np.load(os.path.join(base_path,"data-mean=1-std=3/inversion-vp-baseline/iter_loss.npz"))["data"][:MAX_ITER]
iterloss_baseline4  = np.load(os.path.join(base_path,"data-mean=1-std=4/inversion-vp-baseline/iter_loss.npz"))["data"][:MAX_ITER]
iterloss_baseline5  = np.load(os.path.join(base_path,"data-mean=1-std=5/inversion-vp-baseline/iter_loss.npz"))["data"][:MAX_ITER]
iterloss_baseline6  = np.load(os.path.join(base_path,"data-mean=1-std=6/inversion-vp-baseline/iter_loss.npz"))["data"][:MAX_ITER]

iterloss_1x64_CNN0 = np.load(os.path.join(base_path,"data-mean=0-std=0/inversion-vp-CNN-1x64-v/iter_loss.npz"))["data"][:MAX_ITER]
iterloss_1x64_CNN1 = np.load(os.path.join(base_path,"data-mean=1-std=1/inversion-vp-CNN-1x64-v/iter_loss.npz"))["data"][:MAX_ITER]
iterloss_1x64_CNN2 = np.load(os.path.join(base_path,"data-mean=1-std=2/inversion-vp-CNN-1x64-v/iter_loss.npz"))["data"][:MAX_ITER]
iterloss_1x64_CNN3 = np.load(os.path.join(base_path,"data-mean=1-std=3/inversion-vp-CNN-1x64-v/iter_loss.npz"))["data"][:MAX_ITER]
iterloss_1x64_CNN4 = np.load(os.path.join(base_path,"data-mean=1-std=4/inversion-vp-CNN-1x64-v/iter_loss.npz"))["data"][:MAX_ITER]
iterloss_1x64_CNN5 = np.load(os.path.join(base_path,"data-mean=1-std=5/inversion-vp-CNN-1x64-v/iter_loss.npz"))["data"][:MAX_ITER]
iterloss_1x64_CNN6 = np.load(os.path.join(base_path,"data-mean=1-std=6/inversion-vp-CNN-1x64-v/iter_loss.npz"))["data"][:MAX_ITER]

iterloss_1x128_CNN0 = np.load(os.path.join(base_path,"data-mean=0-std=0/inversion-vp-CNN-1x128-v/iter_loss.npz"))["data"][:MAX_ITER]
iterloss_1x128_CNN1 = np.load(os.path.join(base_path,"data-mean=1-std=1/inversion-vp-CNN-1x128-v/iter_loss.npz"))["data"][:MAX_ITER]
iterloss_1x128_CNN2 = np.load(os.path.join(base_path,"data-mean=1-std=2/inversion-vp-CNN-1x128-v/iter_loss.npz"))["data"][:MAX_ITER]
iterloss_1x128_CNN3 = np.load(os.path.join(base_path,"data-mean=1-std=3/inversion-vp-CNN-1x128-v/iter_loss.npz"))["data"][:MAX_ITER]
iterloss_1x128_CNN4 = np.load(os.path.join(base_path,"data-mean=1-std=4/inversion-vp-CNN-1x128-v/iter_loss.npz"))["data"][:MAX_ITER]
iterloss_1x128_CNN5 = np.load(os.path.join(base_path,"data-mean=1-std=5/inversion-vp-CNN-1x128-v/iter_loss.npz"))["data"][:MAX_ITER]
iterloss_1x128_CNN6 = np.load(os.path.join(base_path,"data-mean=1-std=6/inversion-vp-CNN-1x128-v/iter_loss.npz"))["data"][:MAX_ITER]

iterloss_1x256_CNN0 = np.load(os.path.join(base_path,"data-mean=0-std=0/inversion-vp-CNN-1x256-v/iter_loss.npz"))["data"][:MAX_ITER]
iterloss_1x256_CNN1 = np.load(os.path.join(base_path,"data-mean=1-std=1/inversion-vp-CNN-1x256-v/iter_loss.npz"))["data"][:MAX_ITER]
iterloss_1x256_CNN2 = np.load(os.path.join(base_path,"data-mean=1-std=2/inversion-vp-CNN-1x256-v/iter_loss.npz"))["data"][:MAX_ITER]
iterloss_1x256_CNN3 = np.load(os.path.join(base_path,"data-mean=1-std=3/inversion-vp-CNN-1x256-v/iter_loss.npz"))["data"][:MAX_ITER]
iterloss_1x256_CNN4 = np.load(os.path.join(base_path,"data-mean=1-std=4/inversion-vp-CNN-1x256-v/iter_loss.npz"))["data"][:MAX_ITER]
iterloss_1x256_CNN5 = np.load(os.path.join(base_path,"data-mean=1-std=5/inversion-vp-CNN-1x256-v/iter_loss.npz"))["data"][:MAX_ITER]
iterloss_1x256_CNN6 = np.load(os.path.join(base_path,"data-mean=1-std=6/inversion-vp-CNN-1x256-v/iter_loss.npz"))["data"][:MAX_ITER]

iterloss_2x64_CNN0 = np.load(os.path.join(base_path,"data-mean=0-std=0/inversion-vp-CNN-2x64-v/iter_loss.npz"))["data"][:MAX_ITER]
iterloss_2x64_CNN1 = np.load(os.path.join(base_path,"data-mean=1-std=1/inversion-vp-CNN-2x64-v/iter_loss.npz"))["data"][:MAX_ITER]
iterloss_2x64_CNN2 = np.load(os.path.join(base_path,"data-mean=1-std=2/inversion-vp-CNN-2x64-v/iter_loss.npz"))["data"][:MAX_ITER]
iterloss_2x64_CNN3 = np.load(os.path.join(base_path,"data-mean=1-std=3/inversion-vp-CNN-2x64-v/iter_loss.npz"))["data"][:MAX_ITER]
iterloss_2x64_CNN4 = np.load(os.path.join(base_path,"data-mean=1-std=4/inversion-vp-CNN-2x64-v/iter_loss.npz"))["data"][:MAX_ITER]
iterloss_2x64_CNN5 = np.load(os.path.join(base_path,"data-mean=1-std=5/inversion-vp-CNN-2x64-v/iter_loss.npz"))["data"][:MAX_ITER]
iterloss_2x64_CNN6 = np.load(os.path.join(base_path,"data-mean=1-std=6/inversion-vp-CNN-2x64-v/iter_loss.npz"))["data"][:MAX_ITER]

iterloss_2x128_CNN0 = np.load(os.path.join(base_path,"data-mean=0-std=0/inversion-vp-CNN-2x128-v/iter_loss.npz"))["data"][:MAX_ITER]
iterloss_2x128_CNN1 = np.load(os.path.join(base_path,"data-mean=1-std=1/inversion-vp-CNN-2x128-v/iter_loss.npz"))["data"][:MAX_ITER]
iterloss_2x128_CNN2 = np.load(os.path.join(base_path,"data-mean=1-std=2/inversion-vp-CNN-2x128-v/iter_loss.npz"))["data"][:MAX_ITER]
iterloss_2x128_CNN3 = np.load(os.path.join(base_path,"data-mean=1-std=3/inversion-vp-CNN-2x128-v/iter_loss.npz"))["data"][:MAX_ITER]
iterloss_2x128_CNN4 = np.load(os.path.join(base_path,"data-mean=1-std=4/inversion-vp-CNN-2x128-v/iter_loss.npz"))["data"][:MAX_ITER]
iterloss_2x128_CNN5 = np.load(os.path.join(base_path,"data-mean=1-std=5/inversion-vp-CNN-2x128-v/iter_loss.npz"))["data"][:MAX_ITER]
iterloss_2x128_CNN6 = np.load(os.path.join(base_path,"data-mean=1-std=6/inversion-vp-CNN-2x128-v/iter_loss.npz"))["data"][:MAX_ITER]

iterloss_2x256_CNN0 = np.load(os.path.join(base_path,"data-mean=0-std=0/inversion-vp-CNN-2x256-v/iter_loss.npz"))["data"][:MAX_ITER]
iterloss_2x256_CNN1 = np.load(os.path.join(base_path,"data-mean=1-std=1/inversion-vp-CNN-2x256-v/iter_loss.npz"))["data"][:MAX_ITER]
iterloss_2x256_CNN2 = np.load(os.path.join(base_path,"data-mean=1-std=2/inversion-vp-CNN-2x256-v/iter_loss.npz"))["data"][:MAX_ITER]
iterloss_2x256_CNN3 = np.load(os.path.join(base_path,"data-mean=1-std=3/inversion-vp-CNN-2x256-v/iter_loss.npz"))["data"][:MAX_ITER]
iterloss_2x256_CNN4 = np.load(os.path.join(base_path,"data-mean=1-std=4/inversion-vp-CNN-2x256-v/iter_loss.npz"))["data"][:MAX_ITER]
iterloss_2x256_CNN5 = np.load(os.path.join(base_path,"data-mean=1-std=5/inversion-vp-CNN-2x256-v/iter_loss.npz"))["data"][:MAX_ITER]
iterloss_2x256_CNN6 = np.load(os.path.join(base_path,"data-mean=1-std=6/inversion-vp-CNN-2x256-v/iter_loss.npz"))["data"][:MAX_ITER]

iterloss_3x64_CNN0 = np.load(os.path.join(base_path,"data-mean=0-std=0/inversion-vp-CNN-3x64-v/iter_loss.npz"))["data"][:MAX_ITER]
iterloss_3x64_CNN1 = np.load(os.path.join(base_path,"data-mean=1-std=1/inversion-vp-CNN-3x64-v/iter_loss.npz"))["data"][:MAX_ITER]
iterloss_3x64_CNN2 = np.load(os.path.join(base_path,"data-mean=1-std=2/inversion-vp-CNN-3x64-v/iter_loss.npz"))["data"][:MAX_ITER]
iterloss_3x64_CNN3 = np.load(os.path.join(base_path,"data-mean=1-std=3/inversion-vp-CNN-3x64-v/iter_loss.npz"))["data"][:MAX_ITER]
iterloss_3x64_CNN4 = np.load(os.path.join(base_path,"data-mean=1-std=4/inversion-vp-CNN-3x64-v/iter_loss.npz"))["data"][:MAX_ITER]
iterloss_3x64_CNN5 = np.load(os.path.join(base_path,"data-mean=1-std=5/inversion-vp-CNN-3x64-v/iter_loss.npz"))["data"][:MAX_ITER]
iterloss_3x64_CNN6 = np.load(os.path.join(base_path,"data-mean=1-std=6/inversion-vp-CNN-3x64-v/iter_loss.npz"))["data"][:MAX_ITER]

iterloss_3x128_CNN0 = np.load(os.path.join(base_path,"data-mean=0-std=0/inversion-vp-CNN-3x128-v/iter_loss.npz"))["data"][:MAX_ITER]
iterloss_3x128_CNN1 = np.load(os.path.join(base_path,"data-mean=1-std=1/inversion-vp-CNN-3x128-v/iter_loss.npz"))["data"][:MAX_ITER]
iterloss_3x128_CNN2 = np.load(os.path.join(base_path,"data-mean=1-std=2/inversion-vp-CNN-3x128-v/iter_loss.npz"))["data"][:MAX_ITER]
iterloss_3x128_CNN3 = np.load(os.path.join(base_path,"data-mean=1-std=3/inversion-vp-CNN-3x128-v/iter_loss.npz"))["data"][:MAX_ITER]
iterloss_3x128_CNN4 = np.load(os.path.join(base_path,"data-mean=1-std=4/inversion-vp-CNN-3x128-v/iter_loss.npz"))["data"][:MAX_ITER]
iterloss_3x128_CNN5 = np.load(os.path.join(base_path,"data-mean=1-std=5/inversion-vp-CNN-3x128-v/iter_loss.npz"))["data"][:MAX_ITER]
iterloss_3x128_CNN6 = np.load(os.path.join(base_path,"data-mean=1-std=6/inversion-vp-CNN-3x128-v/iter_loss.npz"))["data"][:MAX_ITER]

iterloss_3x256_CNN0 = np.load(os.path.join(base_path,"data-mean=0-std=0/inversion-vp-CNN-3x256-v/iter_loss.npz"))["data"][:MAX_ITER]
iterloss_3x256_CNN1 = np.load(os.path.join(base_path,"data-mean=1-std=1/inversion-vp-CNN-3x256-v/iter_loss.npz"))["data"][:MAX_ITER]
iterloss_3x256_CNN2 = np.load(os.path.join(base_path,"data-mean=1-std=2/inversion-vp-CNN-3x256-v/iter_loss.npz"))["data"][:MAX_ITER]
iterloss_3x256_CNN3 = np.load(os.path.join(base_path,"data-mean=1-std=3/inversion-vp-CNN-3x256-v/iter_loss.npz"))["data"][:MAX_ITER]
iterloss_3x256_CNN4 = np.load(os.path.join(base_path,"data-mean=1-std=4/inversion-vp-CNN-3x256-v/iter_loss.npz"))["data"][:MAX_ITER]
iterloss_3x256_CNN5 = np.load(os.path.join(base_path,"data-mean=1-std=5/inversion-vp-CNN-3x256-v/iter_loss.npz"))["data"][:MAX_ITER]
iterloss_3x256_CNN6 = np.load(os.path.join(base_path,"data-mean=1-std=6/inversion-vp-CNN-3x256-v/iter_loss.npz"))["data"][:MAX_ITER]

## MAPE

In [ ]:
# calculate the MAPE
WIN_SIZE = 3

baseline_losses = [
    np.min(MAPE(true_v=true_v, inv_v=itervp_baseline0)),
    np.min(MAPE(true_v=true_v, inv_v=itervp_baseline1)),
    np.min(MAPE(true_v=true_v, inv_v=itervp_baseline2)),
    np.min(MAPE(true_v=true_v, inv_v=itervp_baseline3)),
    np.min(MAPE(true_v=true_v, inv_v=itervp_baseline4)),
    np.min(MAPE(true_v=true_v, inv_v=itervp_baseline5)),
    np.min(MAPE(true_v=true_v, inv_v=itervp_baseline6))
]

cnn_losses1x64 = [
    np.min(MAPE(true_v=true_v, inv_v=itervp_CNN_1x64_0)),
    np.min(MAPE(true_v=true_v, inv_v=itervp_CNN_1x64_1)),
    np.min(MAPE(true_v=true_v, inv_v=itervp_CNN_1x64_2)),
    np.min(MAPE(true_v=true_v, inv_v=itervp_CNN_1x64_3)),
    np.min(MAPE(true_v=true_v, inv_v=itervp_CNN_1x64_4)),
    np.min(MAPE(true_v=true_v, inv_v=itervp_CNN_1x64_5)),
    np.min(MAPE(true_v=true_v, inv_v=itervp_CNN_1x64_6))
]

cnn_losses1x128 = [
    np.min(MAPE(true_v=true_v, inv_v=itervp_CNN_1x128_0)),
    np.min(MAPE(true_v=true_v, inv_v=itervp_CNN_1x128_1)),
    np.min(MAPE(true_v=true_v, inv_v=itervp_CNN_1x128_2)),
    np.min(MAPE(true_v=true_v, inv_v=itervp_CNN_1x128_3)),
    np.min(MAPE(true_v=true_v, inv_v=itervp_CNN_1x128_4)),
    np.min(MAPE(true_v=true_v, inv_v=itervp_CNN_1x128_5)),
    np.min(MAPE(true_v=true_v, inv_v=itervp_CNN_1x128_6))
]

cnn_losses1x256 = [
    np.min(MAPE(true_v=true_v, inv_v=itervp_CNN_1x256_0)),
    np.min(MAPE(true_v=true_v, inv_v=itervp_CNN_1x256_1)),
    np.min(MAPE(true_v=true_v, inv_v=itervp_CNN_1x256_2)),
    np.min(MAPE(true_v=true_v, inv_v=itervp_CNN_1x256_3)),
    np.min(MAPE(true_v=true_v, inv_v=itervp_CNN_1x256_4)),
    np.min(MAPE(true_v=true_v, inv_v=itervp_CNN_1x256_5)),
    np.min(MAPE(true_v=true_v, inv_v=itervp_CNN_1x256_6))
]

cnn_losses2x64 = [
    np.min(MAPE(true_v=true_v, inv_v=itervp_CNN_2x64_0)),
    np.min(MAPE(true_v=true_v, inv_v=itervp_CNN_2x64_1)),
    np.min(MAPE(true_v=true_v, inv_v=itervp_CNN_2x64_2)),
    np.min(MAPE(true_v=true_v, inv_v=itervp_CNN_2x64_3)),
    np.min(MAPE(true_v=true_v, inv_v=itervp_CNN_2x64_4)),
    np.min(MAPE(true_v=true_v, inv_v=itervp_CNN_2x64_5)),
    np.min(MAPE(true_v=true_v, inv_v=itervp_CNN_2x64_6))
]

cnn_losses2x128 = [
    np.min(MAPE(true_v=true_v, inv_v=itervp_CNN_2x128_0)),
    np.min(MAPE(true_v=true_v, inv_v=itervp_CNN_2x128_1)),
    np.min(MAPE(true_v=true_v, inv_v=itervp_CNN_2x128_2)),
    np.min(MAPE(true_v=true_v, inv_v=itervp_CNN_2x128_3)),
    np.min(MAPE(true_v=true_v, inv_v=itervp_CNN_2x128_4)),
    np.min(MAPE(true_v=true_v, inv_v=itervp_CNN_2x128_5)),
    np.min(MAPE(true_v=true_v, inv_v=itervp_CNN_2x128_6))
]

cnn_losses2x256 = [
    np.min(MAPE(true_v=true_v, inv_v=itervp_CNN_2x256_0)),
    np.min(MAPE(true_v=true_v, inv_v=itervp_CNN_2x256_1)),
    np.min(MAPE(true_v=true_v, inv_v=itervp_CNN_2x256_2)),
    np.min(MAPE(true_v=true_v, inv_v=itervp_CNN_2x256_3)),
    np.min(MAPE(true_v=true_v, inv_v=itervp_CNN_2x256_4)),
    np.min(MAPE(true_v=true_v, inv_v=itervp_CNN_2x256_5)),
    np.min(MAPE(true_v=true_v, inv_v=itervp_CNN_2x256_6))
]

cnn_losses3x64 = [
    np.min(MAPE(true_v=true_v, inv_v=itervp_CNN_3x64_0)),
    np.min(MAPE(true_v=true_v, inv_v=itervp_CNN_3x64_1)),
    np.min(MAPE(true_v=true_v, inv_v=itervp_CNN_3x64_2)),
    np.min(MAPE(true_v=true_v, inv_v=itervp_CNN_3x64_3)),
    np.min(MAPE(true_v=true_v, inv_v=itervp_CNN_3x64_4)),
    np.min(MAPE(true_v=true_v, inv_v=itervp_CNN_3x64_5)),
    np.min(MAPE(true_v=true_v, inv_v=itervp_CNN_3x64_6))
]

cnn_losses3x128 = [
    np.min(MAPE(true_v=true_v, inv_v=itervp_CNN_3x128_0)),
    np.min(MAPE(true_v=true_v, inv_v=itervp_CNN_3x128_1)),
    np.min(MAPE(true_v=true_v, inv_v=itervp_CNN_3x128_2)),
    np.min(MAPE(true_v=true_v, inv_v=itervp_CNN_3x128_3)),
    np.min(MAPE(true_v=true_v, inv_v=itervp_CNN_3x128_4)),
    np.min(MAPE(true_v=true_v, inv_v=itervp_CNN_3x128_5)),
    np.min(MAPE(true_v=true_v, inv_v=itervp_CNN_3x128_6))
]

cnn_losses3x256 = [
    np.min(MAPE(true_v=true_v, inv_v=itervp_CNN_3x256_0)),
    np.min(MAPE(true_v=true_v, inv_v=itervp_CNN_3x256_1)),
    np.min(MAPE(true_v=true_v, inv_v=itervp_CNN_3x256_2)),
    np.min(MAPE(true_v=true_v, inv_v=itervp_CNN_3x256_3)),
    np.min(MAPE(true_v=true_v, inv_v=itervp_CNN_3x256_4)),
    np.min(MAPE(true_v=true_v, inv_v=itervp_CNN_3x256_5)),
    np.min(MAPE(true_v=true_v, inv_v=itervp_CNN_3x256_6))
]

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

xlist = [0, 1, 2, 3, 4, 5, 6]
xtick_labels = ["0σ₀", "1σ₀", "2σ₀","3σ₀", "4σ₀", "5σ₀", "6σ₀"]

plt.figure(figsize=(6, 4))
plt.plot(xlist, baseline_losses ,  marker='o', linestyle='-',  label="Baseline" , color='k')
plt.plot(xlist, cnn_losses1x64  ,  marker='s', linestyle='--', label="CNN-1x64")
plt.plot(xlist, cnn_losses1x128 ,  marker='s', linestyle='--', label="CNN-1x128")
plt.plot(xlist, cnn_losses1x256 ,  marker='s', linestyle='--', label="CNN-1x256")
plt.plot(xlist, cnn_losses2x64  ,  marker='s', linestyle='--', label="CNN-2x64")
plt.plot(xlist, cnn_losses2x128 ,  marker='s', linestyle='--', label="CNN-2x128")
plt.plot(xlist, cnn_losses2x256 ,  marker='s', linestyle='--', label="CNN-2x256")
plt.plot(xlist, cnn_losses3x64  ,  marker='s', linestyle='--', label="CNN-3x64")
plt.plot(xlist, cnn_losses3x128 ,  marker='s', linestyle='--', label="CNN-3x128")
plt.plot(xlist, cnn_losses3x256 ,  marker='s', linestyle='--', label="CNN-3x256")
plt.xlabel("Noise Level")
plt.ylabel("MAPE Error")
plt.xticks(xlist, xtick_labels)
plt.title("MAPE Error vs. Noise Level")
plt.legend()
plt.grid(True, linestyle="--", alpha=0.6)
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
colors = sns.color_palette("muted")

xlist = np.array([0, 1, 2, 3, 4, 5, 6])
xtick_labels = [r"$0\sigma_0$", r"$1\sigma_0$", r"$2\sigma_0$", 
                r"$3\sigma_0$", r"$4\sigma_0$", r"$5\sigma_0$", r"$6\sigma_0$"]

cnn_losses = np.array([
    cnn_losses1x64, cnn_losses1x128, cnn_losses1x256,
    cnn_losses2x64, cnn_losses2x128, cnn_losses2x256,
    cnn_losses3x64, cnn_losses3x128, cnn_losses3x256
])
cnn_mean = np.mean(cnn_losses, axis=0)
cnn_std = np.std(cnn_losses, axis=0)

plt.figure(figsize=(8, 4))

plt.plot(xlist, baseline_losses, marker='o', linestyle='-', color=colors[0], 
         label="Traditional FWI", linewidth=2.5, markersize=7)

plt.fill_between(xlist, cnn_mean - cnn_std, cnn_mean + cnn_std, color=colors[1], alpha=0.3)
plt.plot(xlist, cnn_mean, marker='s', linestyle='-', color="crimson" , label=f"CNN-$v_p$", linewidth=2.5, markersize=7)
plt.xlabel("Noise Level", fontsize=12)
plt.ylabel("Mean Absolute Percentage Error", fontsize=12)
plt.xticks(xlist, xtick_labels, fontsize=12)
plt.yticks(fontsize=11)
plt.legend(frameon=False, fontsize=12)

plt.grid(True, linestyle="--", alpha=0.6)
# sns.despine(left=False, bottom=False)
plt.show()


## Article Figure (MAPE)

In [ ]:
data_clean  = np.load(os.path.join(base_path,"data-mean=0-std=0/waveform/obs_data.npz"),allow_pickle=True)["data"].item()["p"]
data_noise1 = np.load(os.path.join(base_path,"data-mean=1-std=1/waveform/obs_data.npz"),allow_pickle=True)["data"].item()["p"]
data_noise2 = np.load(os.path.join(base_path,"data-mean=1-std=2/waveform/obs_data.npz"),allow_pickle=True)["data"].item()["p"]
data_noise3 = np.load(os.path.join(base_path,"data-mean=1-std=3/waveform/obs_data.npz"),allow_pickle=True)["data"].item()["p"]
data_noise4 = np.load(os.path.join(base_path,"data-mean=1-std=4/waveform/obs_data.npz"),allow_pickle=True)["data"].item()["p"]
data_noise5 = np.load(os.path.join(base_path,"data-mean=1-std=5/waveform/obs_data.npz"),allow_pickle=True)["data"].item()["p"]
data_noise6 = np.load(os.path.join(base_path,"data-mean=1-std=6/waveform/obs_data.npz"),allow_pickle=True)["data"].item()["p"]

def normalize(data):
    return data/np.max(np.abs(data))

def z_score_normalize(data):
    """
    Z-score 标准化
    """
    return (data - np.mean(data)) / np.std(data)

data_clean  = z_score_normalize(data_clean)
data_noise1 = z_score_normalize(data_noise1)
data_noise2 = z_score_normalize(data_noise2)
data_noise3 = z_score_normalize(data_noise3)
data_noise4 = z_score_normalize(data_noise4)
data_noise5 = z_score_normalize(data_noise5)
data_noise6 = z_score_normalize(data_noise6)

In [ ]:
data_clean.shape

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import gridspec
import seaborn as sns
plt.rcParams['svg.fonttype'] = 'none'
sns.set_style("ticks")


fig = plt.figure(figsize=(15, 10))
gs = gridspec.GridSpec(3, 28,height_ratios=[1,1.2,1])

################################################################
axes = [fig.add_subplot(gs[0, i*4:(i+1)*4]) for i in range(7)]

shot_num = 20

resample_rate_t = 5
resample_rate_x = 1
nx_plot = nx//resample_rate_x
nt_plot = nt//resample_rate_t
dt_plot = dt*resample_rate_t

# Create meshgrid for waveform visualization
z_mesh_waveform, x_mesh_waveform = np.meshgrid(np.arange(nx_plot), np.arange(nt_plot) * dt_plot)

# List of waveform data and titles
data_list = [
    data_clean[shot_num,::resample_rate_t,::resample_rate_x],
    data_noise1[shot_num,::resample_rate_t,::resample_rate_x],
    data_noise2[shot_num,::resample_rate_t,::resample_rate_x],
    data_noise3[shot_num,::resample_rate_t,::resample_rate_x],
    data_noise4[shot_num,::resample_rate_t,::resample_rate_x],
    data_noise5[shot_num,::resample_rate_t,::resample_rate_x],
    data_noise6[shot_num,::resample_rate_t,::resample_rate_x],
]

titles = [r"Clean", r"$\mathcal{N} \sim (\mu_0, 1\sigma_0)$", r"$\mathcal{N} \sim (\mu_0, 2\sigma_0)$", r"$\mathcal{N} \sim (\mu_0, 3\sigma_0)$", r"$\mathcal{N} \sim (\mu_0, 4\sigma_0)$", r"$\mathcal{N} \sim (\mu_0, 5\sigma_0)$", r"$\mathcal{N} \sim (\mu_0, 6\sigma_0)$"]


# Plot waveforms
for i, ax in enumerate(axes):
    vmin, vmax = (-3, 3)   # Different color scales for clean and noisy data
    pcm = ax.pcolormesh(z_mesh_waveform, x_mesh_waveform, data_list[i], cmap="gray", vmin=vmin, vmax=vmax)
    
    ax.invert_yaxis()
    ax.set_title(titles[i], fontsize=14)
    
    # Remove yticks except for the first subplot
    if i > 0:
        ax.set_yticks([])
    
    # Only add ylabel to the first subplot
    if i == 0:
        ax.set_ylabel("Time (s)", fontsize=14)
    ax.tick_params(labelsize=14)
    # ax.set_xticks([])
# Add a single xlabel at the bottom center
# fig.text(0.5, -0.01, "Receivers", ha="center", fontsize=14)

##################################################################

palette = sns.color_palette("deep")
colors = {
    "baseline": "#4D4D4D",  # Dark gray
    "cnn_dv": palette[0],    # Blue for CNN-v
    "cnn_v": sns.dark_palette(palette[0], 2)[1],  # Darker blue for CNN-dv
    "mlp_dv": palette[1],    # Green for MLP-v
    "mlp_v": sns.dark_palette(palette[1], 2)[1],  # Darker green for MLP-dv
    "unet_dv": palette[2],   # Purple for Unet-v
    "unet_v": sns.dark_palette(palette[2], 2)[1]  # Darker purple for Unet-dv
}

axes1 = fig.add_subplot(gs[1, :])

xlist = np.array([0, 1, 2, 3, 4, 5, 6])
xtick_labels = [r"$0\sigma_0$", r"$1\sigma_0$", r"$2\sigma_0$", r"$3\sigma_0$", r"$4\sigma_0$", r"$5\sigma_0$", r"$6\sigma_0$"]

cnn_losses = np.array([
    cnn_losses1x64, cnn_losses1x128, cnn_losses1x256,
    cnn_losses2x64, cnn_losses2x128, cnn_losses2x256,
    cnn_losses3x64, cnn_losses3x128, cnn_losses3x256
])
cnn_mean = np.mean(cnn_losses, axis=0)
cnn_std  = np.std(cnn_losses, axis=0)


axes1.plot(xlist, baseline_losses, linestyle='-', marker='o',color=colors['baseline'], label="Traditional FWI", linewidth=2.5, markersize=7)
axes1.fill_between(xlist, cnn_mean - cnn_std, cnn_mean + cnn_std, color=colors['cnn_v'], alpha=0.3)
axes1.plot(xlist, cnn_mean,  linestyle='-', marker='s',color=colors['cnn_v'] , label=f"CNN-$v_p$", linewidth=2.5, markersize=7)

# axes1.set_xlabel("Noise Level", fontsize=14)
axes1.set_ylabel("MAPE", fontsize=14)

axes1.set_xticks(xlist, xtick_labels, fontsize=14)
axes1.tick_params(labelsize=14)
axes1.set_ylim(3.5, 8.5)
axes1.legend(frameon=False, fontsize=14,loc='upper left')

axes1.grid(True, linestyle="--", alpha=0.6)


##################################################################

axes21 = fig.add_subplot(gs[2, 1:13])
axes22 = fig.add_subplot(gs[2, 15:-1])

im1 = plot_vel_single_for_all(fig,axes21,itervp_baseline6[-1]      ,title=None   ,MSE=r"Traditonal FWI MAPE:" + " " + r"{:.2f}".format(MAPE(true_v,itervp_baseline6[-1])))
im2 = plot_vel_single_for_all(fig,axes22,itervp_CNN_3x256_6[-1]    ,title=None   ,MSE=r"CNN-$v_p$ MAPE:" + "\t" + r"{:.2f}".format(MAPE(true_v,itervp_CNN_3x256_6[-1])))


axes21.set_xlabel("Distance (km)", fontsize=14)
axes22.set_xlabel("Distance (km)", fontsize=14)
axes21.set_ylabel("Depth (km)", fontsize=14)
axes22.set_ylabel("Depth (km)", fontsize=14)

from matplotlib.ticker import MaxNLocator
axes21.xaxis.set_major_locator(MaxNLocator(7))
axes22.xaxis.set_major_locator(MaxNLocator(7))


fig.text(0.1, 0.89, "(a)", fontsize=16, fontweight='bold', ha="center", va="center")
fig.text(0.1, 0.62, "(b)", fontsize=16, fontweight='bold', ha="center", va="center")
fig.text(0.12, 0.31, "(c)", fontsize=16, fontweight='bold', ha="center", va="center")
fig.text(0.485, 0.31, "(d)", fontsize=16, fontweight='bold', ha="center", va="center")

# Adjust layout for better spacing
plt.subplots_adjust(hspace=0.3, wspace=0.1, right=0.85)

import matplotlib as mpl
def add_bottom_cax(ax, pad, height,shrink=1):
    axpos = ax.get_position()
    width = axpos.x1 - axpos.x0
    left_position = axpos.x0 + width * (1 - shrink) / 2
    caxpos = mpl.transforms.Bbox.from_extents(
        left_position,
        axpos.y0 - pad,
        left_position + width * shrink,
        axpos.y0 - pad + height
    )
    cax = ax.figure.add_axes(caxpos)
    return cax

# add horizontal colorbar for [axes21, axes22]
cbar_ax1 = add_bottom_cax(axes21, 0.08, 0.02,shrink=0.8)
cbar1 = fig.colorbar(im1, cax=cbar_ax1, orientation='horizontal', pad=0.1, shrink=0.95)
cbar1.ax.tick_params(labelsize=14)
cbar1.ax.text(1.02, 0.5, r'$m/s$', fontsize=14, transform=cbar1.ax.transAxes, 
              verticalalignment='center', horizontalalignment='left')

cbar_ax2 = add_bottom_cax(axes22, 0.08, 0.02,shrink=0.8)
cbar2 = fig.colorbar(im2, cax=cbar_ax2, orientation='horizontal', pad=0.1, shrink=0.95)
cbar2.ax.tick_params(labelsize=14)
cbar2.ax.text(1.02, 0.5, r'$m/s$', fontsize=14, transform=cbar2.ax.transAxes, 
              verticalalignment='center', horizontalalignment='left')

# plt.savefig("./Figures/Figure6_Noise_Tests_on_Marmousi2_MAPE.png",bbox_inches='tight',dpi=300)
# plt.savefig("./Figures_PDF/Figure6_Noise_Tests_on_Marmousi2_MAPE.pdf",bbox_inches='tight',format="pdf",dpi=300)
plt.savefig("./Figures_SVG/Figure6_Noise_Tests_on_Marmousi2_MAPE.svg",bbox_inches='tight',format="svg")

plt.show()

## SSIM

In [ ]:
# 计算 MAPE 误差
WIN_SIZE = 3

baseline_losses = [
    np.max(SSIM(true_v=true_v, inv_v=itervp_baseline0,win_size=WIN_SIZE)),
    np.max(SSIM(true_v=true_v, inv_v=itervp_baseline1,win_size=WIN_SIZE)),
    np.max(SSIM(true_v=true_v, inv_v=itervp_baseline2,win_size=WIN_SIZE)),
    np.max(SSIM(true_v=true_v, inv_v=itervp_baseline3,win_size=WIN_SIZE)),
    np.max(SSIM(true_v=true_v, inv_v=itervp_baseline4,win_size=WIN_SIZE)),
    np.max(SSIM(true_v=true_v, inv_v=itervp_baseline5,win_size=WIN_SIZE)),
    np.max(SSIM(true_v=true_v, inv_v=itervp_baseline6,win_size=WIN_SIZE))
]

cnn_losses1x64 = [
    np.max(SSIM(true_v=true_v, inv_v=itervp_CNN_1x64_0,win_size=WIN_SIZE)),
    np.max(SSIM(true_v=true_v, inv_v=itervp_CNN_1x64_1,win_size=WIN_SIZE)),
    np.max(SSIM(true_v=true_v, inv_v=itervp_CNN_1x64_2,win_size=WIN_SIZE)),
    np.max(SSIM(true_v=true_v, inv_v=itervp_CNN_1x64_3,win_size=WIN_SIZE)),
    np.max(SSIM(true_v=true_v, inv_v=itervp_CNN_1x64_4,win_size=WIN_SIZE)),
    np.max(SSIM(true_v=true_v, inv_v=itervp_CNN_1x64_5,win_size=WIN_SIZE)),
    np.max(SSIM(true_v=true_v, inv_v=itervp_CNN_1x64_6,win_size=WIN_SIZE))
]

cnn_losses1x128 = [
    np.max(SSIM(true_v=true_v, inv_v=itervp_CNN_1x128_0,win_size=WIN_SIZE)),
    np.max(SSIM(true_v=true_v, inv_v=itervp_CNN_1x128_1,win_size=WIN_SIZE)),
    np.max(SSIM(true_v=true_v, inv_v=itervp_CNN_1x128_2,win_size=WIN_SIZE)),
    np.max(SSIM(true_v=true_v, inv_v=itervp_CNN_1x128_3,win_size=WIN_SIZE)),
    np.max(SSIM(true_v=true_v, inv_v=itervp_CNN_1x128_4,win_size=WIN_SIZE)),
    np.max(SSIM(true_v=true_v, inv_v=itervp_CNN_1x128_5,win_size=WIN_SIZE)),
    np.max(SSIM(true_v=true_v, inv_v=itervp_CNN_1x128_6,win_size=WIN_SIZE))
]

cnn_losses1x256 = [
    np.max(SSIM(true_v=true_v, inv_v=itervp_CNN_1x256_0,win_size=WIN_SIZE)),
    np.max(SSIM(true_v=true_v, inv_v=itervp_CNN_1x256_1,win_size=WIN_SIZE)),
    np.max(SSIM(true_v=true_v, inv_v=itervp_CNN_1x256_2,win_size=WIN_SIZE)),
    np.max(SSIM(true_v=true_v, inv_v=itervp_CNN_1x256_3,win_size=WIN_SIZE)),
    np.max(SSIM(true_v=true_v, inv_v=itervp_CNN_1x256_4,win_size=WIN_SIZE)),
    np.max(SSIM(true_v=true_v, inv_v=itervp_CNN_1x256_5,win_size=WIN_SIZE)),
    np.max(SSIM(true_v=true_v, inv_v=itervp_CNN_1x256_6,win_size=WIN_SIZE))
]

cnn_losses2x64 = [
    np.max(SSIM(true_v=true_v, inv_v=itervp_CNN_2x64_0,win_size=WIN_SIZE)),
    np.max(SSIM(true_v=true_v, inv_v=itervp_CNN_2x64_1,win_size=WIN_SIZE)),
    np.max(SSIM(true_v=true_v, inv_v=itervp_CNN_2x64_2,win_size=WIN_SIZE)),
    np.max(SSIM(true_v=true_v, inv_v=itervp_CNN_2x64_3,win_size=WIN_SIZE)),
    np.max(SSIM(true_v=true_v, inv_v=itervp_CNN_2x64_4,win_size=WIN_SIZE)),
    np.max(SSIM(true_v=true_v, inv_v=itervp_CNN_2x64_5,win_size=WIN_SIZE)),
    np.max(SSIM(true_v=true_v, inv_v=itervp_CNN_2x64_6,win_size=WIN_SIZE))
]

cnn_losses2x128 = [
    np.max(SSIM(true_v=true_v, inv_v=itervp_CNN_2x128_0,win_size=WIN_SIZE)),
    np.max(SSIM(true_v=true_v, inv_v=itervp_CNN_2x128_1,win_size=WIN_SIZE)),
    np.max(SSIM(true_v=true_v, inv_v=itervp_CNN_2x128_2,win_size=WIN_SIZE)),
    np.max(SSIM(true_v=true_v, inv_v=itervp_CNN_2x128_3,win_size=WIN_SIZE)),
    np.max(SSIM(true_v=true_v, inv_v=itervp_CNN_2x128_4,win_size=WIN_SIZE)),
    np.max(SSIM(true_v=true_v, inv_v=itervp_CNN_2x128_5,win_size=WIN_SIZE)),
    np.max(SSIM(true_v=true_v, inv_v=itervp_CNN_2x128_6,win_size=WIN_SIZE))
]

cnn_losses2x256 = [
    np.max(SSIM(true_v=true_v, inv_v=itervp_CNN_2x256_0,win_size=WIN_SIZE)),
    np.max(SSIM(true_v=true_v, inv_v=itervp_CNN_2x256_1,win_size=WIN_SIZE)),
    np.max(SSIM(true_v=true_v, inv_v=itervp_CNN_2x256_2,win_size=WIN_SIZE)),
    np.max(SSIM(true_v=true_v, inv_v=itervp_CNN_2x256_3,win_size=WIN_SIZE)),
    np.max(SSIM(true_v=true_v, inv_v=itervp_CNN_2x256_4,win_size=WIN_SIZE)),
    np.max(SSIM(true_v=true_v, inv_v=itervp_CNN_2x256_5,win_size=WIN_SIZE)),
    np.max(SSIM(true_v=true_v, inv_v=itervp_CNN_2x256_6,win_size=WIN_SIZE))
]

cnn_losses3x64 = [
    np.max(SSIM(true_v=true_v, inv_v=itervp_CNN_3x64_0,win_size=WIN_SIZE)),
    np.max(SSIM(true_v=true_v, inv_v=itervp_CNN_3x64_1,win_size=WIN_SIZE)),
    np.max(SSIM(true_v=true_v, inv_v=itervp_CNN_3x64_2,win_size=WIN_SIZE)),
    np.max(SSIM(true_v=true_v, inv_v=itervp_CNN_3x64_3,win_size=WIN_SIZE)),
    np.max(SSIM(true_v=true_v, inv_v=itervp_CNN_3x64_4,win_size=WIN_SIZE)),
    np.max(SSIM(true_v=true_v, inv_v=itervp_CNN_3x64_5,win_size=WIN_SIZE)),
    np.max(SSIM(true_v=true_v, inv_v=itervp_CNN_3x64_6,win_size=WIN_SIZE))
]

cnn_losses3x128 = [
    np.max(SSIM(true_v=true_v, inv_v=itervp_CNN_3x128_0,win_size=WIN_SIZE)),
    np.max(SSIM(true_v=true_v, inv_v=itervp_CNN_3x128_1,win_size=WIN_SIZE)),
    np.max(SSIM(true_v=true_v, inv_v=itervp_CNN_3x128_2,win_size=WIN_SIZE)),
    np.max(SSIM(true_v=true_v, inv_v=itervp_CNN_3x128_3,win_size=WIN_SIZE)),
    np.max(SSIM(true_v=true_v, inv_v=itervp_CNN_3x128_4,win_size=WIN_SIZE)),
    np.max(SSIM(true_v=true_v, inv_v=itervp_CNN_3x128_5,win_size=WIN_SIZE)),
    np.max(SSIM(true_v=true_v, inv_v=itervp_CNN_3x128_6,win_size=WIN_SIZE))
]

cnn_losses3x256 = [
    np.max(SSIM(true_v=true_v, inv_v=itervp_CNN_3x256_0,win_size=WIN_SIZE)),
    np.max(SSIM(true_v=true_v, inv_v=itervp_CNN_3x256_1,win_size=WIN_SIZE)),
    np.max(SSIM(true_v=true_v, inv_v=itervp_CNN_3x256_2,win_size=WIN_SIZE)),
    np.max(SSIM(true_v=true_v, inv_v=itervp_CNN_3x256_3,win_size=WIN_SIZE)),
    np.max(SSIM(true_v=true_v, inv_v=itervp_CNN_3x256_4,win_size=WIN_SIZE)),
    np.max(SSIM(true_v=true_v, inv_v=itervp_CNN_3x256_5,win_size=WIN_SIZE)),
    np.max(SSIM(true_v=true_v, inv_v=itervp_CNN_3x256_6,win_size=WIN_SIZE))
]

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# sns.set_theme(style="whitegrid")
colors = sns.color_palette("muted")

xlist = np.array([0, 1, 2, 3, 4, 5, 6])
xtick_labels = [r"$0\sigma_0$", r"$1\sigma_0$", r"$2\sigma_0$", 
                r"$3\sigma_0$", r"$4\sigma_0$", r"$5\sigma_0$", r"$6\sigma_0$"]

cnn_losses = np.array([
    cnn_losses1x64, cnn_losses1x128, cnn_losses1x256,
    cnn_losses2x64, cnn_losses2x128, cnn_losses2x256,
    cnn_losses3x64, cnn_losses3x128, cnn_losses3x256
])

cnn_mean = np.mean(cnn_losses, axis=0)
cnn_std = np.std(cnn_losses, axis=0)

plt.figure(figsize=(8, 4))

plt.plot(xlist, baseline_losses, marker='o', linestyle='-', color=colors[0], 
         label="Traditional FWI", linewidth=1.5, markersize=7)

plt.fill_between(xlist, cnn_mean - cnn_std, cnn_mean + cnn_std, color=colors[1], alpha=0.3)
plt.plot(xlist, cnn_mean, marker='s', linestyle='-', color="crimson" , label=f"CNN-$v_p$", linewidth=1.5, markersize=7)

plt.xlabel("Noise Level", fontsize=12)
plt.ylabel("Structural Similarity Index Measure", fontsize=12)

plt.xticks(xlist, xtick_labels, fontsize=12)
plt.yticks(fontsize=11)

plt.legend(frameon=False, fontsize=12)

plt.grid(True, linestyle="--", alpha=0.6)


plt.show()


## Article Figure: SSIM

In [ ]:
data_clean  = np.load(os.path.join(base_path,"data-mean=0-std=0/waveform/obs_data.npz"),allow_pickle=True)["data"].item()["p"]
data_noise1 = np.load(os.path.join(base_path,"data-mean=1-std=1/waveform/obs_data.npz"),allow_pickle=True)["data"].item()["p"]
data_noise2 = np.load(os.path.join(base_path,"data-mean=1-std=2/waveform/obs_data.npz"),allow_pickle=True)["data"].item()["p"]
data_noise3 = np.load(os.path.join(base_path,"data-mean=1-std=3/waveform/obs_data.npz"),allow_pickle=True)["data"].item()["p"]
data_noise4 = np.load(os.path.join(base_path,"data-mean=1-std=4/waveform/obs_data.npz"),allow_pickle=True)["data"].item()["p"]
data_noise5 = np.load(os.path.join(base_path,"data-mean=1-std=5/waveform/obs_data.npz"),allow_pickle=True)["data"].item()["p"]
data_noise6 = np.load(os.path.join(base_path,"data-mean=1-std=6/waveform/obs_data.npz"),allow_pickle=True)["data"].item()["p"]
def normalize(data):
    return data/np.max(np.abs(data))

data_clean = normalize(data_clean)
data_noise1 = normalize(data_noise1)
data_noise2 = normalize(data_noise2)
data_noise3 = normalize(data_noise3)
data_noise4 = normalize(data_noise4)
data_noise5 = normalize(data_noise5)
data_noise6 = normalize(data_noise6)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import gridspec
import seaborn as sns
sns.set_style("ticks")  # 或者使用 "darkgrid", "ticks", "white", "dark"


# Create the figure and define grid layout
fig = plt.figure(figsize=(15, 10))
gs = gridspec.GridSpec(3, 28,height_ratios=[1,1.2,1])

################################################################
axes = [fig.add_subplot(gs[0, i*4:(i+1)*4]) for i in range(7)]

shot_num = 20

# Create meshgrid for waveform visualization
z_mesh_waveform, x_mesh_waveform = np.meshgrid(np.arange(nx), np.arange(nt) * dt)

# List of waveform data and titles
data_list = [
    data_clean[shot_num],
    data_noise1[shot_num],
    data_noise2[shot_num],
    data_noise3[shot_num],
    data_noise4[shot_num],
    data_noise5[shot_num],
    data_noise6[shot_num],
]

titles = [r"clean", r"$\mathcal{N} \sim (\mu_0, 1\sigma_0)$", r"$\mathcal{N} \sim (\mu_0, 2\sigma_0)$", r"$\mathcal{N} \sim (\mu_0, 3\sigma_0)$", r"$\mathcal{N} \sim (\mu_0, 4\sigma_0)$", r"$\mathcal{N} \sim (\mu_0, 5\sigma_0)$", r"$\mathcal{N} \sim (\mu_0, 6\sigma_0)$"]


# Plot waveforms
for i, ax in enumerate(axes):
    vmin, vmax = (-0.1, 0.1) if i == 0 else (-0.15, 0.15)  # Different color scales for clean and noisy data
    pcm = ax.pcolormesh(z_mesh_waveform, x_mesh_waveform, data_list[i], cmap="gray", vmin=vmin, vmax=vmax)
    
    ax.invert_yaxis()
    ax.set_title(titles[i], fontsize=14)
    
    # Remove yticks except for the first subplot
    if i > 0:
        ax.set_yticks([])
    
    # Only add ylabel to the first subplot
    if i == 0:
        ax.set_ylabel("Time (s)", fontsize=14)
    ax.tick_params(labelsize=14)
    # ax.set_xticks([])
# Add a single xlabel at the bottom center
# fig.text(0.5, -0.01, "Receivers", ha="center", fontsize=14)

##################################################################

palette = sns.color_palette("deep")
colors = {
    "baseline": "#4D4D4D",  # Dark gray
    "cnn_dv": palette[0],    # Blue for CNN-v
    "cnn_v": sns.dark_palette(palette[0], 2)[1],  # Darker blue for CNN-dv
    "mlp_dv": palette[1],    # Green for MLP-v
    "mlp_v": sns.dark_palette(palette[1], 2)[1],  # Darker green for MLP-dv
    "unet_dv": palette[2],   # Purple for Unet-v
    "unet_v": sns.dark_palette(palette[2], 2)[1]  # Darker purple for Unet-dv
}

axes1 = fig.add_subplot(gs[1, :])

xlist = np.array([0, 1, 2, 3, 4, 5, 6])
xtick_labels = [r"$0\sigma_0$", r"$1\sigma_0$", r"$2\sigma_0$", r"$3\sigma_0$", r"$4\sigma_0$", r"$5\sigma_0$", r"$6\sigma_0$"]

cnn_losses = np.array([
    cnn_losses1x64, cnn_losses1x128, cnn_losses1x256,
    cnn_losses2x64, cnn_losses2x128, cnn_losses2x256,
    cnn_losses3x64, cnn_losses3x128, cnn_losses3x256
])
cnn_mean = np.mean(cnn_losses, axis=0)
cnn_std  = np.std(cnn_losses, axis=0)


axes1.plot(xlist, baseline_losses, linestyle='-', marker='o',color=colors['baseline'], label="Traditional FWI", linewidth=2.5, markersize=7)
axes1.fill_between(xlist, cnn_mean - cnn_std, cnn_mean + cnn_std, color=colors['cnn_v'], alpha=0.3)
axes1.plot(xlist, cnn_mean,  linestyle='-', marker='s',color=colors['cnn_v'] , label=f"CNN-$v_p$", linewidth=2.5, markersize=7)

# axes1.set_xlabel("Noise Level", fontsize=14)
axes1.set_ylabel("SSIM", fontsize=14)

axes1.set_xticks(xlist, xtick_labels, fontsize=14)
axes1.tick_params(labelsize=14)

axes1.legend(frameon=False, fontsize=14,loc='lower left')

axes1.grid(True, linestyle="--", alpha=0.6)

##################################################################

axes21 = fig.add_subplot(gs[2, 1:13])
axes22 = fig.add_subplot(gs[2, 15:-1])

im1 = plot_vel_single_for_all(fig,axes21,itervp_baseline6[-1]      ,title=None   ,MSE=r"Baseline  SSIM:" + " " + r"{:.2f}".format(SSIM(true_v,itervp_baseline6[-1],win_size=WIN_SIZE)))
im2 = plot_vel_single_for_all(fig,axes22,itervp_CNN_3x256_6[-1]    ,title=None   ,MSE=r"CNN-$v_p$ SSIM:" + "\t" + r"{:.2f}".format(SSIM(true_v,itervp_CNN_3x256_6[-1],win_size=WIN_SIZE)))


axes21.set_xlabel("Distance (km)", fontsize=14)
axes22.set_xlabel("Distance (km)", fontsize=14)
axes21.set_ylabel("Depth (km)", fontsize=14)
axes22.set_ylabel("Depth (km)", fontsize=14)

from matplotlib.ticker import MaxNLocator
axes21.xaxis.set_major_locator(MaxNLocator(7))
axes22.xaxis.set_major_locator(MaxNLocator(7))


fig.text(0.08, 0.89, "(a)", fontsize=14, fontweight='bold', ha="center", va="center")
fig.text(0.08, 0.62, "(b)", fontsize=14, fontweight='bold', ha="center", va="center")
fig.text(0.12, 0.31, "(c)", fontsize=14, fontweight='bold', ha="center", va="center")
fig.text(0.485, 0.31, "(d)", fontsize=14, fontweight='bold', ha="center", va="center")

# Adjust layout for better spacing
plt.subplots_adjust(hspace=0.3, wspace=0.1, right=0.85)

# Adjust layout for better spacing
plt.subplots_adjust(hspace=0.3, wspace=0.1, right=0.85)

import matplotlib as mpl
def add_bottom_cax(ax, pad, height,shrink=1):
    axpos = ax.get_position()
    width = axpos.x1 - axpos.x0
    left_position = axpos.x0 + width * (1 - shrink) / 2
    caxpos = mpl.transforms.Bbox.from_extents(
        left_position,
        axpos.y0 - pad,
        left_position + width * shrink,
        axpos.y0 - pad + height
    )
    cax = ax.figure.add_axes(caxpos)
    return cax

# add horizontal colorbar for [axes21, axes22]
cbar_ax1 = add_bottom_cax(axes21, 0.08, 0.02,shrink=0.8)
cbar1 = fig.colorbar(im1, cax=cbar_ax1, orientation='horizontal', pad=0.1, shrink=0.8)
cbar1.ax.tick_params(labelsize=14)
cbar1.ax.text(1.02, 0.5, r'$m/s$', fontsize=14, transform=cbar1.ax.transAxes, 
              verticalalignment='center', horizontalalignment='left')

cbar_ax2 = add_bottom_cax(axes22, 0.08, 0.02,shrink=0.8)
cbar2 = fig.colorbar(im2, cax=cbar_ax2, orientation='horizontal', pad=0.1, shrink=0.8)
cbar2.ax.tick_params(labelsize=14)
cbar2.ax.text(1.02, 0.5, r'$m/s$', fontsize=14, transform=cbar2.ax.transAxes, 
              verticalalignment='center', horizontalalignment='left')

plt.show()